# Inference Engine — Colab

Runs the C++/CUDA engine end to end on a T4. **No training needed**: the engine
is brought up against randomly initialized weights, because numerical parity does
not care whether the model is any good.

Unlike the older `colab.ipynb`, this notebook does **not** paste source files
inline. It clones the repos. There are ~20 source files now, and duplicating them
into `%%writefile` cells guarantees the notebook drifts out of sync with the repo.

Runtime -> Change runtime type -> **T4 GPU** before running.

## 0. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| available", torch.cuda.is_available())

## 1. Clone both repos

The engine needs both: `export_weights.py` lives in **wikitext-gpt**, the engine
itself lives in **fused-attention**. (This split is the main practical argument
for merging them into one repo.)

In [ ]:
# Branches: the engine lives on `engine` / `engine-export` until parity passes
# here and it merges to main. After merging, drop the -b flags.
GPT_BRANCH    = 'engine-export'
ENGINE_BRANCH = 'engine'

# Which export to run against. This is the ONLY line to change between passes:
#   pass 1 (bring-up, random weights) -> '/content/export'
#   pass 3 (after training)           -> '/content/drive/MyDrive/wikitext-gpt/export'
EXPORT = '/content/export'

if EXPORT.startswith('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')

%cd /content
!rm -rf wikitext-gpt fused-attention
!git clone -q -b {GPT_BRANCH}    https://github.com/williamclymire-tamu/wikitext-gpt.git
!git clone -q -b {ENGINE_BRANCH} https://github.com/williamclymire-tamu/fused-attention.git

# sanity: these files only exist on the engine branch
!test -f fused-attention/transformer.cu && echo "OK engine branch" || echo "WRONG BRANCH - no transformer.cu"
!test -f wikitext-gpt/export_weights.py && echo "OK gpt branch"    || echo "WRONG BRANCH - no export_weights.py"

## 2. Export random weights

`--random` skips the checkpoint and exports a freshly initialized model, so the
engine can be validated before training finishes. Writes flat fp32 `.bin` files
plus per-stage activation fixtures captured from PyTorch.

In [ ]:
%cd /content/wikitext-gpt
import os
if EXPORT == '/content/export':
    !python export_weights.py --random --out {EXPORT}
else:
    print('pass 3: using the trained export already at', EXPORT)
!ls {EXPORT} | head
!ls {EXPORT}/parity

## 3. Validate the wiring on CPU first

`test_cpu` builds with plain g++ — no CUDA, no GPU. It checks the scalar C++
reference against the PyTorch fixtures stage by stage, then checks that
incremental KV-cache decode reproduces a full prefill.

If this fails, the bug is in the forward-pass wiring and no amount of staring at
CUDA will help. Expect `ALL PASS`.

In [ ]:
%cd /content/fused-attention
!make test_cpu
!./test_cpu {EXPORT}

## 4. Build the engine

`HEAD_DIM` is compile-time. It must match `head_dim` in `config.json`
(d_model 256 / n_head 4 = 64). The engine checks this at startup and tells you to
rebuild rather than reading past the head boundary and emitting plausible garbage.

In [ ]:
!make engine ARCH=sm_75 HEAD_DIM=64

## 5. Parity — the one that matters

Same prompt PyTorch ran, diffed at every stage. Tolerances loosen deliberately
after the embedding: `--use_fast_math` changes `expf`/`tanhf` and cuBLAS reorders
accumulation, so some drift is expected. Drift that *grows sharply* at one layer
is the signal worth chasing.

In [ ]:
!./engine parity {EXPORT}

## 6. Decode throughput

In [ ]:
!./engine bench {EXPORT} --tokens 256

## 7. Generation smoke test

Output is gibberish here — the weights are random and `--random` export has no
token table, so tokens print as `[id]`. This only proves the sampling loop and
the KV cache advance correctly. Real text comes after training.

In [ ]:
!./engine generate {EXPORT} --ids 3,4,5 --tokens 20 --temp 0.8 --top-k 40

## 8. Optional — the original kernel benchmarks

The standalone prefill/decode kernel tests that predate the engine.

In [ ]:
!python generate_reference.py && python generate_day1_ref.py
!make attention test_day1 ARCH=sm_75
!./attention test test_data
!./test_day1 test test_day1_data
!./attention bench
!./test_day1 bench